In [3]:
# Jupyter cell: Validate a DataFrame with columns ["ID", "Phenotype"] (and optionally ["Weight"]),
# loaded and saved via pickle

# ------------------- Config -------------------
INPUT_PICKLE = "/work/gr-fe/bryan/data/SHCS/01_raw/phenotype.pkl"
OUTPUT_PICKLE = "/work/gr-fe/bryan/data/SHCS/02_processed/phenotype.processed.pkl"
SAVE_CLEANED = True
STRICT_COLS = False                     # If True, fail when unexpected columns exist (see below)
EXPECTED_ID = "ID"                      # Expected ID column name (case-insensitive)
EXPECTED_PHENOTYPE = "Phenotype"        # Expected Phenotype column name (case-insensitive)

WEIGHT = True                           # <---- NEW: allow/require Weight column
EXPECTED_WEIGHT = "Weight"              # Expected Weight column name (case-insensitive)

# ------------------- Imports -------------------
import pandas as pd
import numpy as np
from pathlib import Path

# ------------------- Load -------------------
obj = pd.read_pickle(INPUT_PICKLE)['meta']
if not isinstance(obj, pd.DataFrame):
    raise TypeError(f"Pickle did not contain a pandas DataFrame (got {type(obj)}).")

df_raw = obj.copy()
print("Loaded DataFrame shape:", df_raw.shape)
display(df_raw.head(3))

# ------------------- Validation / Cleaning -------------------
def validate_id_phenotype(
    df: pd.DataFrame,
    id_col=EXPECTED_ID,
    pheno_col=EXPECTED_PHENOTYPE,
    weight: bool = False,
    weight_col=EXPECTED_WEIGHT,
    strict_cols: bool = False,
):
    report = {"ok": True, "messages": []}
    df = df.copy()

    # 1) Find required columns (case-insensitive exact match)
    lower_map = {c.lower(): c for c in df.columns}

    required = {
        id_col.lower(): "ID",
        pheno_col.lower(): "Phenotype",
    }
    if weight:
        required[weight_col.lower()] = "Weight"

    missing = [orig for key, orig in required.items() if key not in lower_map]
    if missing:
        report["ok"] = False
        report["messages"].append(
            f"Missing required columns: {missing}. Columns found: {list(df.columns)}"
        )
        return report, df

    # 2) Rename to canonical names
    rename_map = {lower_map[k]: v for k, v in required.items()}
    df = df.rename(columns=rename_map)

    # 3) Optional: enforce expected set of columns
    expected_set = {"ID", "Phenotype"} | ({"Weight"} if weight else set())
    if strict_cols and set(df.columns) != expected_set:
        report["ok"] = False
        report["messages"].append(
            f"Expected exactly columns {sorted(expected_set)}, got: {list(df.columns)}"
        )
        return report, df

    # 4) Keep only relevant columns for cleaned output
    keep_cols = ["ID", "Phenotype"] + (["Weight"] if weight else [])
    core = df[keep_cols].copy()

    # 5) Normalize: strip whitespace and stray quotes (ID/Phenotype only)
    def _clean_str(s):
        if pd.isna(s):
            return np.nan
        s = str(s).strip()
        s = s.strip('"').strip("'")
        return s.strip()

    core["ID"] = core["ID"].map(_clean_str)
    core["Phenotype"] = core["Phenotype"].map(_clean_str)

    # If weight is requested, coerce to numeric
    if weight:
        core["Weight"] = pd.to_numeric(core["Weight"], errors="coerce")

    # 6) Basic checks
    n_rows = len(core)
    n_missing_id = core["ID"].isna().sum()
    n_missing_ph = core["Phenotype"].isna().sum()

    if n_missing_id > 0:
        report["ok"] = False
        report["messages"].append(f"Found {n_missing_id} rows with missing ID.")
    if n_missing_ph > 0:
        report["ok"] = False
        report["messages"].append(f"Found {n_missing_ph} rows with missing Phenotype.")

    if weight:
        n_missing_w = core["Weight"].isna().sum()
        if n_missing_w > 0:
            # choose whether this should fail or just warn; currently FAIL to match other missing checks
            report["ok"] = False
            report["messages"].append(
                f"Found {n_missing_w} rows with missing/unparseable Weight (after numeric coercion)."
            )

    # 7) Duplicates
    # If Weight is present, define "duplicate" as same (ID, Phenotype) pair and keep the first weight.
    dup_pair = core.duplicated(["ID", "Phenotype"]).sum()
    if dup_pair > 0:
        report["messages"].append(
            f"Found {dup_pair} duplicate (ID, Phenotype) pairs. These will be dropped in cleaned output (keeping first)."
        )

    # 8) Summary stats
    n_ids = core["ID"].nunique(dropna=True)
    n_ph = core["Phenotype"].nunique(dropna=True)
    report["messages"].append(f"Rows: {n_rows}, unique IDs: {n_ids}, unique Phenotypes: {n_ph}")
    if weight:
        report["messages"].append(
            f"Weight summary (non-missing): n={core['Weight'].notna().sum()}, "
            f"min={core['Weight'].min()}, median={core['Weight'].median()}, max={core['Weight'].max()}"
        )

    # 9) Create cleaned version
    core_clean = core.drop_duplicates(["ID", "Phenotype"], keep="first").reset_index(drop=True)

    return report, core_clean


report, df_clean = validate_id_phenotype(
    df_raw,
    id_col=EXPECTED_ID,
    pheno_col=EXPECTED_PHENOTYPE,
    weight=WEIGHT,
    weight_col=EXPECTED_WEIGHT,
    strict_cols=STRICT_COLS,
)

print("OK:", report["ok"])
for m in report["messages"]:
    print("-", m)

print("Cleaned shape:", df_clean.shape)
display(df_clean.head(5))

# ------------------- Save (pickle) -------------------
if SAVE_CLEANED:
    out_path = OUTPUT_PICKLE
    if out_path is None:
        p = Path(INPUT_PICKLE)
        out_path = p.with_name(p.stem + ".processed.pkl")
    df_clean.to_pickle(out_path)
    print(f"Cleaned DataFrame saved to: {out_path}")


Loaded DataFrame shape: (2708, 3)


,ID,Phenotype,Weight
2795,30119,CAD,122
3237,31593,CAD,556
2235,25890,CAD,54


OK: True
- Rows: 2708, unique IDs: 2095, unique Phenotypes: 5
- Weight summary (non-missing): n=2708, min=-12998, median=437.5, max=9497
Cleaned shape: (2708, 3)


,ID,Phenotype,Weight
0,30119,CAD,122
1,31593,CAD,556
2,25890,CAD,54
3,47196,CAD,225
4,17150,CAD,118


Cleaned DataFrame saved to: /work/gr-fe/bryan/data/SHCS/02_processed/phenotype.processed.pkl
